In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

ipo_returns_df = pd.read_csv(
    "../data/processed/ipo_returns_analysis.csv"
)

date_columns = [
    "LISTING DATE",
    "6M ACTUAL DATE",
    "12M ACTUAL DATE"
]

for col in date_columns:
    ipo_returns_df[col] = pd.to_datetime(
        ipo_returns_df[col],
        errors="coerce"
    )

print("IPO dataset loaded:", ipo_returns_df.shape)

IPO dataset loaded: (194, 16)


In [4]:
# Find the date range required for the benchmark
start_date = ipo_returns_df["LISTING DATE"].min()

end_dates = pd.concat([
    ipo_returns_df["6M ACTUAL DATE"],
    ipo_returns_df["12M ACTUAL DATE"]
]).dropna()

end_date = end_dates.max()

print("Benchmark start date:", start_date)
print("Benchmark last observation date:", end_date)

# Download Nifty 50 historical data
nifty_df = yf.download(
    "^NSEI",
    start=start_date.strftime("%Y-%m-%d"),
    end=(end_date + pd.Timedelta(days=5)).strftime("%Y-%m-%d"),
    auto_adjust=False,
    progress=False
)

print("\nDownloaded Nifty 50 data")
print("Shape:", nifty_df.shape)

display(nifty_df.head())
display(nifty_df.tail())

Benchmark start date: 2022-11-15 00:00:00
Benchmark last observation date: 2026-10-26 00:00:00

Downloaded Nifty 50 data
Shape: (978, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,,
2022-11-15,18403.400391,18403.400391,18427.949219,18282.000000,18362.750000,250900
2022-11-16,18409.650391,18409.650391,18442.150391,18344.150391,18398.250000,219300
2022-11-17,18343.900391,18343.900391,18417.599609,18312.949219,18358.699219,200500
2022-11-18,18307.650391,18307.650391,18394.599609,18209.800781,18382.949219,198800
2022-11-21,18159.949219,18159.949219,18262.300781,18133.349609,18246.400391,213800


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,,
2026-10-26,22753.199219,22753.199219,22753.199219,22387.349609,22524.400391,310200
2026-10-27,22749.750000,22749.750000,22829.750000,22636.849609,22708.800781,400900
2026-10-28,22698.849609,22698.849609,22778.050781,22497.699219,22678.000000,295000
2026-10-29,22936.750000,22936.750000,22961.949219,22656.750000,22690.650391,344000
2026-10-30,23053.099609,23053.099609,23118.099609,22905.099609,22964.500000,328600


In [5]:
print("Column structure:")
print(nifty_df.columns)

print("\nData types:")
print(nifty_df.dtypes)

print("\nMissing values:")
print(nifty_df.isnull().sum())

Column structure:
MultiIndex([('Adj Close', '^NSEI'),
            (    'Close', '^NSEI'),
            (     'High', '^NSEI'),
            (      'Low', '^NSEI'),
            (     'Open', '^NSEI'),
            (   'Volume', '^NSEI')],
           names=['Price', 'Ticker'])

Data types:
Price      Ticker
Adj Close  ^NSEI     float64
Close      ^NSEI     float64
High       ^NSEI     float64
Low        ^NSEI     float64
Open       ^NSEI     float64
Volume     ^NSEI       int64
dtype: object

Missing values:
Price      Ticker
Adj Close  ^NSEI     0
Close      ^NSEI     0
High       ^NSEI     0
Low        ^NSEI     0
Open       ^NSEI     0
Volume     ^NSEI     0
dtype: int64


In [6]:
# Flatten Yahoo Finance MultiIndex columns
nifty_df.columns = nifty_df.columns.get_level_values(0)

# Keep only closing price
nifty_df = nifty_df[["Close"]].copy()

# Convert index into a date column
nifty_df = nifty_df.reset_index()

# Rename columns
nifty_df.columns = ["DATE", "NIFTY_CLOSE"]

# Ensure date format
nifty_df["DATE"] = pd.to_datetime(nifty_df["DATE"])

# Sort by date
nifty_df = nifty_df.sort_values("DATE").reset_index(drop=True)

print("Cleaned Nifty dataset shape:", nifty_df.shape)

display(nifty_df.head())
display(nifty_df.tail())

Cleaned Nifty dataset shape: (978, 2)


,DATE,NIFTY_CLOSE
0,2022-11-15,18403.400391
1,2022-11-16,18409.650391
2,2022-11-17,18343.900391
3,2022-11-18,18307.650391
4,2022-11-21,18159.949219


,DATE,NIFTY_CLOSE
973,2026-10-26,22753.199219
974,2026-10-27,22749.750000
975,2026-10-28,22698.849609
976,2026-10-29,22936.750000
977,2026-10-30,23053.099609


In [7]:
# Ensure sorted benchmark dates
nifty_df = nifty_df.sort_values("DATE").reset_index(drop=True)

nifty_dates = nifty_df["DATE"].values
nifty_closes = nifty_df["NIFTY_CLOSE"].values


def get_nifty_price(target_date):

    if pd.isna(target_date):
        return pd.Series({
            "NIFTY_ACTUAL_DATE": pd.NaT,
            "NIFTY_CLOSE": np.nan
        })

    target_date = pd.Timestamp(target_date)

    position = np.searchsorted(
        nifty_dates,
        np.datetime64(target_date),
        side="left"
    )

    if position >= len(nifty_dates):
        return pd.Series({
            "NIFTY_ACTUAL_DATE": pd.NaT,
            "NIFTY_CLOSE": np.nan
        })

    return pd.Series({
        "NIFTY_ACTUAL_DATE": nifty_dates[position],
        "NIFTY_CLOSE": nifty_closes[position]
    })

In [8]:
test_dates = [
    "2022-11-15",
    "2023-05-21",
    "2023-05-22",
    "2023-05-23"
]

for date in test_dates:

    result = get_nifty_price(date)

    print(
        date,
        "->",
        result["NIFTY_ACTUAL_DATE"],
        "->",
        result["NIFTY_CLOSE"]
    )

2022-11-15 -> 2022-11-15T00:00:00 -> 18403.400390625
2023-05-21 -> 2023-05-22T00:00:00 -> 18314.400390625
2023-05-22 -> 2023-05-22T00:00:00 -> 18314.400390625
2023-05-23 -> 2023-05-23T00:00:00 -> 18348.0


In [9]:
benchmark_df = ipo_returns_df.copy()

# Match listing date
listing_prices = benchmark_df["LISTING DATE"].apply(
    get_nifty_price
)

listing_prices.columns = [
    "NIFTY LISTING ACTUAL DATE",
    "NIFTY LISTING CLOSE"
]

# Match 6-month observation date
six_month_prices = benchmark_df["6M ACTUAL DATE"].apply(
    get_nifty_price
)

six_month_prices.columns = [
    "NIFTY 6M ACTUAL DATE",
    "NIFTY 6M CLOSE"
]

# Match 12-month observation date
twelve_month_prices = benchmark_df["12M ACTUAL DATE"].apply(
    get_nifty_price
)

twelve_month_prices.columns = [
    "NIFTY 12M ACTUAL DATE",
    "NIFTY 12M CLOSE"
]

# Combine benchmark prices
benchmark_df = pd.concat(
    [
        benchmark_df,
        listing_prices,
        six_month_prices,
        twelve_month_prices
    ],
    axis=1
)

display(
    benchmark_df[
        [
            "COMPANY NAME",
            "LISTING DATE",
            "NIFTY LISTING ACTUAL DATE",
            "NIFTY LISTING CLOSE",
            "6M ACTUAL DATE",
            "NIFTY 6M ACTUAL DATE",
            "NIFTY 6M CLOSE",
            "12M ACTUAL DATE",
            "NIFTY 12M ACTUAL DATE",
            "NIFTY 12M CLOSE"
        ]
    ].head(10)
)

,COMPANY NAME,LISTING DATE,NIFTY LISTING ACTUAL DATE,NIFTY LISTING CLOSE,6M ACTUAL DATE,NIFTY 6M ACTUAL DATE,NIFTY 6M CLOSE,12M ACTUAL DATE,NIFTY 12M ACTUAL DATE,NIFTY 12M CLOSE
0,Fusion Micro Finance Limited,2022-11-15,2022-11-15,18403.400391,2023-05-15,2023-05-15,18398.849609,2023-11-15,2023-11-15,19675.449219
1,Global Health Limited,2022-11-16,2022-11-16,18409.650391,2023-05-16,2023-05-16,18286.500000,2023-11-16,2023-11-16,19765.199219
2,Bikaji Foods International Limited,2022-11-16,2022-11-16,18409.650391,2023-05-16,2023-05-16,18286.500000,2023-11-16,2023-11-16,19765.199219
3,Archean Chemical Industries Limited,2022-11-21,2022-11-21,18159.949219,2023-05-22,2023-05-22,18314.400391,2023-11-21,2023-11-21,19783.400391
4,FiveStar Business Finance Limited,2022-11-21,2022-11-21,18159.949219,2023-05-22,2023-05-22,18314.400391,2023-11-21,2023-11-21,19783.400391
5,Kaynes Technology India Limited,2022-11-22,2022-11-22,18244.199219,2023-05-22,2023-05-22,18314.400391,2023-11-22,2023-11-22,19811.849609
6,Inox Green Energy Services Limited,2022-11-23,2022-11-23,18267.250000,2023-05-23,2023-05-23,18348.000000,2023-11-23,2023-11-23,19802.000000
7,Keystone Realtors Limited,2022-11-24,2022-11-24,18484.099609,2023-05-24,2023-05-24,18285.400391,2023-11-24,2023-11-24,19794.699219
8,Dharmaj Crop Guard Limited,2022-12-08,2022-12-08,18609.349609,2023-06-08,2023-06-08,18634.550781,2023-12-08,2023-12-08,20969.400391
9,Uniparts India Limited,2022-12-12,2022-12-12,18497.150391,2023-06-12,2023-06-12,18601.500000,2023-12-12,2023-12-12,20906.400391


In [10]:
# Nifty listing-to-6M return
benchmark_df["NIFTY 6M RETURN (%)"] = (
    (
        benchmark_df["NIFTY 6M CLOSE"]
        - benchmark_df["NIFTY LISTING CLOSE"]
    )
    / benchmark_df["NIFTY LISTING CLOSE"]
) * 100


# Nifty listing-to-12M return
benchmark_df["NIFTY 12M RETURN (%)"] = (
    (
        benchmark_df["NIFTY 12M CLOSE"]
        - benchmark_df["NIFTY LISTING CLOSE"]
    )
    / benchmark_df["NIFTY LISTING CLOSE"]
) * 100


display(
    benchmark_df[
        [
            "COMPANY NAME",
            "NIFTY LISTING CLOSE",
            "NIFTY 6M RETURN (%)",
            "NIFTY 12M RETURN (%)"
        ]
    ].head(10)
)

,COMPANY NAME,NIFTY LISTING CLOSE,NIFTY 6M RETURN (%),NIFTY 12M RETURN (%)
0,Fusion Micro Finance Limited,18403.400391,-0.024728,6.912031
1,Global Health Limited,18409.650391,-0.668945,7.363251
2,Bikaji Foods International Limited,18409.650391,-0.668945,7.363251
3,Archean Chemical Industries Limited,18159.949219,0.850504,8.939734
4,FiveStar Business Finance Limited,18159.949219,0.850504,8.939734
5,Kaynes Technology India Limited,18244.199219,0.384786,8.592596
6,Inox Green Energy Services Limited,18267.250000,0.442048,8.401648
7,Keystone Realtors Limited,18484.099609,-1.074974,7.090416
8,Dharmaj Crop Guard Limited,18609.349609,0.135422,12.682070
9,Uniparts India Limited,18497.150391,0.564139,13.024979


In [11]:
# Check missing values in benchmark columns

benchmark_columns = [
    "NIFTY LISTING ACTUAL DATE",
    "NIFTY LISTING CLOSE",
    "NIFTY 6M ACTUAL DATE",
    "NIFTY 6M CLOSE",
    "NIFTY 12M ACTUAL DATE",
    "NIFTY 12M CLOSE",
    "NIFTY 6M RETURN (%)",
    "NIFTY 12M RETURN (%)"
]

missing_summary = benchmark_df[benchmark_columns].isnull().sum()

print("Missing Values in Nifty Benchmark Data")
display(missing_summary.to_frame(name="Missing Count"))

print("Benchmark dataset shape:", benchmark_df.shape)

# Check for invalid or zero prices
print(
    "Zero or negative Nifty prices:",
    (benchmark_df[
        ["NIFTY LISTING CLOSE", "NIFTY 6M CLOSE", "NIFTY 12M CLOSE"]
    ] <= 0).sum()
)

Missing Values in Nifty Benchmark Data


,Missing Count
NIFTY LISTING ACTUAL DATE,0
NIFTY LISTING CLOSE,0
NIFTY 6M ACTUAL DATE,4
NIFTY 6M CLOSE,4
NIFTY 12M ACTUAL DATE,4
NIFTY 12M CLOSE,4
NIFTY 6M RETURN (%),4
NIFTY 12M RETURN (%),4


Benchmark dataset shape: (194, 24)
Zero or negative Nifty prices: NIFTY LISTING CLOSE    0
NIFTY 6M CLOSE         0
NIFTY 12M CLOSE        0
dtype: int64


In [12]:
# Identify IPOs with missing Nifty benchmark prices

missing_benchmark_df = benchmark_df[
    benchmark_df["NIFTY 6M CLOSE"].isna()
    | benchmark_df["NIFTY 12M CLOSE"].isna()
]

display(
    missing_benchmark_df[
        [
            "COMPANY NAME",
            "LISTING DATE",
            "6M ACTUAL DATE",
            "12M ACTUAL DATE",
            "NIFTY 6M ACTUAL DATE",
            "NIFTY 12M ACTUAL DATE"
        ]
    ]
)

print("Number of IPOs with missing benchmark data:",
      len(missing_benchmark_df))

,COMPANY NAME,LISTING DATE,6M ACTUAL DATE,12M ACTUAL DATE,NIFTY 6M ACTUAL DATE,NIFTY 12M ACTUAL DATE
11,Abans Holdings Limited,2022-12-23,NaT,NaT,NaT,NaT
134,Standard Glass Lining Technology Limited,2025-01-13,NaT,NaT,NaT,NaT
146,Arisinfra Solutions Limited,2025-06-25,NaT,NaT,NaT,NaT
177,Saatvik Green Energy Limited,2025-09-26,NaT,NaT,NaT,NaT


Number of IPOs with missing benchmark data: 4


In [13]:
# Nifty 50 benchmark performance summary

nifty_summary = []

for period in ["6M", "12M"]:

    col = f"NIFTY {period} RETURN (%)"

    returns = benchmark_df[col].dropna()

    nifty_summary.append({
        "Holding Period": period,
        "Available IPOs": len(returns),
        "Average Return (%)": returns.mean(),
        "Median Return (%)": returns.median(),
        "Positive Returns": (returns > 0).sum(),
        "Negative Returns": (returns < 0).sum(),
        "Positive Return (%)": (returns > 0).mean() * 100,
        "Negative Return (%)": (returns < 0).mean() * 100
    })

nifty_summary_df = pd.DataFrame(nifty_summary)

display(nifty_summary_df.round(2))

,Holding Period,Available IPOs,Average Return (%),Median Return (%),Positive Returns,Negative Returns,Positive Return (%),Negative Return (%)
0,6M,190,4.40,4.56,140,50,73.68,26.32
1,12M,190,7.68,6.98,131,59,68.95,31.05


In [14]:
for col in ipo_returns_df.columns:
    print(col)

COMPANY NAME
YF_TICKER
ISSUE PRICE
LISTING DATE
LISTING CLOSE
LISTING RETURN (%)
6M TARGET DATE
6M ACTUAL DATE
6M CLOSE
6M RETURN (%)
12M TARGET DATE
12M ACTUAL DATE
12M CLOSE
12M RETURN (%)
6M DELAY DAYS
12M DELAY DAYS


In [15]:
print("Benchmark columns:")
print(benchmark_df.columns.tolist())

Benchmark columns:
['COMPANY NAME', 'YF_TICKER', 'ISSUE PRICE', 'LISTING DATE', 'LISTING CLOSE', 'LISTING RETURN (%)', '6M TARGET DATE', '6M ACTUAL DATE', '6M CLOSE', '6M RETURN (%)', '12M TARGET DATE', '12M ACTUAL DATE', '12M CLOSE', '12M RETURN (%)', '6M DELAY DAYS', '12M DELAY DAYS', 'NIFTY LISTING ACTUAL DATE', 'NIFTY LISTING CLOSE', 'NIFTY 6M ACTUAL DATE', 'NIFTY 6M CLOSE', 'NIFTY 12M ACTUAL DATE', 'NIFTY 12M CLOSE', 'NIFTY 6M RETURN (%)', 'NIFTY 12M RETURN (%)']


In [16]:
# Calculate returns from listing close to 6M and 12M

comparison_df = benchmark_df.copy()

comparison_df["LISTING TO 6M RETURN (%)"] = (
    (comparison_df["6M CLOSE"] - comparison_df["LISTING CLOSE"])
    / comparison_df["LISTING CLOSE"]
) * 100

comparison_df["LISTING TO 12M RETURN (%)"] = (
    (comparison_df["12M CLOSE"] - comparison_df["LISTING CLOSE"])
    / comparison_df["LISTING CLOSE"]
) * 100

display(
    comparison_df[
        [
            "COMPANY NAME",
            "LISTING CLOSE",
            "6M CLOSE",
            "LISTING TO 6M RETURN (%)",
            "12M CLOSE",
            "LISTING TO 12M RETURN (%)"
        ]
    ].head(10).round(2)
)

,COMPANY NAME,LISTING CLOSE,6M CLOSE,LISTING TO 6M RETURN (%),12M CLOSE,LISTING TO 12M RETURN (%)
0,Fusion Micro Finance Limited,324.80,450.70,38.76,597.15,83.85
1,Global Health Limited,NaN,572.50,NaN,888.50,NaN
2,Bikaji Foods International Limited,317.45,369.85,16.51,549.10,72.97
3,Archean Chemical Industries Limited,458.15,542.85,18.49,525.05,14.60
4,FiveStar Business Finance Limited,490.00,529.35,8.03,803.60,64.00
5,Kaynes Technology India Limited,689.95,1306.95,89.43,2491.15,261.06
6,Inox Green Energy Services Limited,59.10,46.85,-20.73,68.95,16.67
7,Keystone Realtors Limited,556.85,509.10,-8.58,536.65,-3.63
8,Dharmaj Crop Guard Limited,266.30,174.50,-34.47,262.90,-1.28
9,Uniparts India Limited,539.65,567.75,5.21,549.95,1.91


In [17]:
# Calculate excess returns over Nifty 50

comparison_df["6M ISSUE VS NIFTY"] = (
    comparison_df["6M RETURN (%)"]
    - comparison_df["NIFTY 6M RETURN (%)"]
)

comparison_df["6M LISTING VS NIFTY"] = (
    comparison_df["LISTING TO 6M RETURN (%)"]
    - comparison_df["NIFTY 6M RETURN (%)"]
)

comparison_df["12M ISSUE VS NIFTY"] = (
    comparison_df["12M RETURN (%)"]
    - comparison_df["NIFTY 12M RETURN (%)"]
)

comparison_df["12M LISTING VS NIFTY"] = (
    comparison_df["LISTING TO 12M RETURN (%)"]
    - comparison_df["NIFTY 12M RETURN (%)"]
)

display(
    comparison_df[
        [
            "COMPANY NAME",
            "6M RETURN (%)",
            "NIFTY 6M RETURN (%)",
            "6M ISSUE VS NIFTY",
            "LISTING TO 6M RETURN (%)",
            "6M LISTING VS NIFTY",
            "12M RETURN (%)",
            "NIFTY 12M RETURN (%)",
            "12M ISSUE VS NIFTY",
            "LISTING TO 12M RETURN (%)",
            "12M LISTING VS NIFTY"
        ]
    ].head(10).round(2)
)

,COMPANY NAME,6M RETURN (%),NIFTY 6M RETURN (%),6M ISSUE VS NIFTY,LISTING TO 6M RETURN (%),6M LISTING VS NIFTY,12M RETURN (%),NIFTY 12M RETURN (%),12M ISSUE VS NIFTY,LISTING TO 12M RETURN (%),12M LISTING VS NIFTY
0,Fusion Micro Finance Limited,22.47,-0.02,22.50,38.76,38.79,62.27,6.91,55.36,83.85,76.94
1,Global Health Limited,70.39,-0.67,71.06,NaN,NaN,164.43,7.36,157.07,NaN,NaN
2,Bikaji Foods International Limited,23.28,-0.67,23.95,16.51,17.18,83.03,7.36,75.67,72.97,65.61
3,Archean Chemical Industries Limited,33.38,0.85,32.53,18.49,17.64,29.00,8.94,20.07,14.60,5.66
4,FiveStar Business Finance Limited,11.68,0.85,10.83,8.03,7.18,69.54,8.94,60.60,64.00,55.06
5,Kaynes Technology India Limited,122.65,0.38,122.26,89.43,89.04,324.39,8.59,315.79,261.06,252.47
6,Inox Green Energy Services Limited,-27.92,0.44,-28.37,-20.73,-21.17,6.08,8.40,-2.32,16.67,8.27
7,Keystone Realtors Limited,-5.90,-1.07,-4.82,-8.58,-7.50,-0.80,7.09,-7.89,-3.63,-10.72
8,Dharmaj Crop Guard Limited,-26.37,0.14,-26.51,-34.47,-34.61,10.93,12.68,-1.75,-1.28,-13.96
9,Uniparts India Limited,-1.60,0.56,-2.17,5.21,4.64,-4.69,13.02,-17.71,1.91,-11.12


In [18]:
# Check missing prices required for IPO return calculations

price_columns = [
    "LISTING CLOSE",
    "6M CLOSE",
    "12M CLOSE"
]

print("Missing price counts:")
display(comparison_df[price_columns].isna().sum().to_frame("Missing Count"))

# Identify affected IPOs
missing_price_df = comparison_df[
    comparison_df[price_columns].isna().any(axis=1)
]

display(
    missing_price_df[
        [
            "COMPANY NAME",
            "LISTING DATE",
            "LISTING CLOSE",
            "6M CLOSE",
            "12M CLOSE"
        ]
    ]
)

print("Total affected IPOs:", len(missing_price_df))

Missing price counts:


,Missing Count
LISTING CLOSE,9
6M CLOSE,4
12M CLOSE,4


,COMPANY NAME,LISTING DATE,LISTING CLOSE,6M CLOSE,12M CLOSE
1,Global Health Limited,2022-11-16,NaN,572.500000,888.500000
11,Abans Holdings Limited,2022-12-23,NaN,NaN,NaN
124,International Gemmological Institute (India) L...,2024-12-20,NaN,345.950012,321.149994
134,Standard Glass Lining Technology Limited,2025-01-13,NaN,NaN,NaN
146,Arisinfra Solutions Limited,2025-06-25,NaN,NaN,NaN
177,Saatvik Green Energy Limited,2025-09-26,NaN,NaN,NaN
184,Epack Prefab Technologies Limited,2025-10-01,NaN,138.770004,242.070007
185,Jain Resource Recycling Limited,2025-10-01,NaN,464.700012,282.500000
186,TruAlt Bioenergy Limited,2025-10-03,NaN,421.450012,442.299988


Total affected IPOs: 9


In [19]:
# Data availability summary

data_quality_summary = pd.DataFrame({
    "Metric": [
        "Total IPOs",
        "Available Listing Close",
        "Missing Listing Close",
        "Available 6M Close",
        "Missing 6M Close",
        "Available 12M Close",
        "Missing 12M Close"
    ],
    "Count": [
        len(comparison_df),
        comparison_df["LISTING CLOSE"].notna().sum(),
        comparison_df["LISTING CLOSE"].isna().sum(),
        comparison_df["6M CLOSE"].notna().sum(),
        comparison_df["6M CLOSE"].isna().sum(),
        comparison_df["12M CLOSE"].notna().sum(),
        comparison_df["12M CLOSE"].isna().sum()
    ]
})

data_quality_summary["Percentage (%)"] = (
    data_quality_summary["Count"] / len(comparison_df) * 100
)

display(data_quality_summary.round(2))

,Metric,Count,Percentage (%)
0,Total IPOs,194,100.00
1,Available Listing Close,185,95.36
2,Missing Listing Close,9,4.64
3,Available 6M Close,190,97.94
4,Missing 6M Close,4,2.06
5,Available 12M Close,190,97.94
6,Missing 12M Close,4,2.06


In [20]:
# Calculate matched sample sizes and performance

strategies = {
    "6M Issue Price": (
        "6M RETURN (%)",
        "NIFTY 6M RETURN (%)"
    ),
    "6M Listing Close": (
        "LISTING TO 6M RETURN (%)",
        "NIFTY 6M RETURN (%)"
    ),
    "12M Issue Price": (
        "12M RETURN (%)",
        "NIFTY 12M RETURN (%)"
    ),
    "12M Listing Close": (
        "LISTING TO 12M RETURN (%)",
        "NIFTY 12M RETURN (%)"
    )
}

comparison_summary = []

for strategy, (ipo_col, nifty_col) in strategies.items():

    matched = comparison_df[[ipo_col, nifty_col]].dropna()

    excess_return = matched[ipo_col] - matched[nifty_col]

    comparison_summary.append({
        "Strategy": strategy,
        "Matched IPOs": len(matched),
        "IPO Average Return (%)": matched[ipo_col].mean(),
        "Nifty Average Return (%)": matched[nifty_col].mean(),
        "Average Excess Return (pp)": excess_return.mean(),
        "Median Excess Return (pp)": excess_return.median(),
        "IPOs Outperforming Nifty": (excess_return > 0).sum(),
        "Outperformance (%)": (excess_return > 0).mean() * 100
    })

comparison_summary_df = pd.DataFrame(comparison_summary)

display(comparison_summary_df.round(2))

,Strategy,Matched IPOs,IPO Average Return (%),Nifty Average Return (%),Average Excess Return (pp),Median Excess Return (pp),IPOs Outperforming Nifty,Outperformance (%)
0,6M Issue Price,190,23.36,4.40,18.96,7.37,107,56.32
1,6M Listing Close,185,5.49,4.63,0.86,-5.58,78,42.16
2,12M Issue Price,190,35.61,7.68,27.93,5.64,101,53.16
3,12M Listing Close,185,15.65,7.94,7.71,-9.40,76,41.08


In [21]:
from scipy.stats import wilcoxon, binomtest

statistical_results = []

strategies = {
    "6M Issue Price": "6M ISSUE VS NIFTY",
    "6M Listing Close": "6M LISTING VS NIFTY",
    "12M Issue Price": "12M ISSUE VS NIFTY",
    "12M Listing Close": "12M LISTING VS NIFTY"
}

for strategy, column in strategies.items():

    excess = comparison_df[column].dropna()

    # Wilcoxon signed-rank test against zero
    wilcoxon_result = wilcoxon(
        excess,
        alternative="two-sided",
        zero_method="wilcox"
    )

    # Binomial test: Is outperformance frequency different from 50%?
    wins = (excess > 0).sum()
    total = len(excess)

    binomial_result = binomtest(
        wins,
        total,
        p=0.5,
        alternative="two-sided"
    )

    statistical_results.append({
        "Strategy": strategy,
        "Sample Size": total,
        "Mean Excess (%)": excess.mean(),
        "Median Excess (%)": excess.median(),
        "Wilcoxon Statistic": wilcoxon_result.statistic,
        "Wilcoxon P-value": wilcoxon_result.pvalue,
        "Outperformance Count": wins,
        "Outperformance Rate (%)": wins / total * 100,
        "Binomial P-value": binomial_result.pvalue
    })

statistical_results_df = pd.DataFrame(statistical_results)

display(statistical_results_df.round(4))

,Strategy,Sample Size,Mean Excess (%),Median Excess (%),Wilcoxon Statistic,Wilcoxon P-value,Outperformance Count,Outperformance Rate (%),Binomial P-value
0,6M Issue Price,190,18.9577,7.3740,6577.0,0.0010,107,56.3158,0.0949
1,6M Listing Close,185,0.8617,-5.5779,7507.0,0.1331,78,42.1622,0.0392
2,12M Issue Price,190,27.9348,5.6413,6755.0,0.0023,101,53.1579,0.4249
3,12M Listing Close,185,7.7069,-9.4014,7951.0,0.3717,76,41.0811,0.0184


In [22]:
# Export IPO vs Nifty 50 benchmark dataset

export_columns = [
    "COMPANY NAME",
    "YF_TICKER",
    "ISSUE PRICE",
    "LISTING DATE",
    "LISTING CLOSE",
    "LISTING RETURN (%)",
    "6M CLOSE",
    "6M RETURN (%)",
    "12M CLOSE",
    "12M RETURN (%)",
    "NIFTY LISTING CLOSE",
    "NIFTY 6M CLOSE",
    "NIFTY 12M CLOSE",
    "NIFTY 6M RETURN (%)",
    "NIFTY 12M RETURN (%)",
    "LISTING TO 6M RETURN (%)",
    "LISTING TO 12M RETURN (%)",
    "6M ISSUE VS NIFTY",
    "6M LISTING VS NIFTY",
    "12M ISSUE VS NIFTY",
    "12M LISTING VS NIFTY"
]

powerbi_df = comparison_df[export_columns].copy()

powerbi_df.to_csv(
    "../data/processed/ipo_nifty_benchmark_analysis.csv",
    index=False
)

print("Benchmark CSV exported successfully")
print("Dataset shape:", powerbi_df.shape)

display(powerbi_df.head())

Benchmark CSV exported successfully
Dataset shape: (194, 21)


,COMPANY NAME,YF_TICKER,ISSUE PRICE,LISTING DATE,LISTING CLOSE,LISTING RETURN (%),6M CLOSE,6M RETURN (%),12M CLOSE,12M RETURN (%),...,NIFTY 6M CLOSE,NIFTY 12M CLOSE,NIFTY 6M RETURN (%),NIFTY 12M RETURN (%),LISTING TO 6M RETURN (%),LISTING TO 12M RETURN (%),6M ISSUE VS NIFTY,6M LISTING VS NIFTY,12M ISSUE VS NIFTY,12M LISTING VS NIFTY
0,Fusion Micro Finance Limited,FUSION.NS,368.0,2022-11-15,324.799988,-11.739134,450.700012,22.472829,597.150024,62.269028,...,18398.849609,19675.449219,-0.024728,6.912031,38.762324,83.851615,22.497557,38.787052,55.356997,76.939584
1,Global Health Limited,MEDANTA.NS,336.0,2022-11-16,NaN,NaN,572.500000,70.386905,888.500000,164.434524,...,18286.500000,19765.199219,-0.668945,7.363251,NaN,NaN,71.055850,NaN,157.071272,NaN
2,Bikaji Foods International Limited,BIKAJI.NS,300.0,2022-11-16,317.450012,5.816671,369.850006,23.283335,549.099976,83.033325,...,18286.500000,19765.199219,-0.668945,7.363251,16.506534,72.972107,23.952280,17.175479,75.670074,65.608856
3,Archean Chemical Industries Limited,ACI.NS,407.0,2022-11-21,458.149994,12.567566,542.849976,33.378372,525.049988,29.004911,...,18314.400391,19783.400391,0.850504,8.939734,18.487391,14.602203,32.527868,17.636887,20.065177,5.662469
4,FiveStar Business Finance Limited,FIVESTAR.NS,474.0,2022-11-21,490.000000,3.375527,529.349976,11.677210,803.599976,69.535860,...,18314.400391,19783.400391,0.850504,8.939734,8.030607,63.999995,10.826706,7.180103,60.596126,55.060261


In [23]:
statistical_results_df.to_csv(
    "../data/processed/ipo_nifty_statistical_tests.csv",
    index=False
)

print("Statistical test CSV exported successfully")
display(statistical_results_df)

Statistical test CSV exported successfully


,Strategy,Sample Size,Mean Excess (%),Median Excess (%),Wilcoxon Statistic,Wilcoxon P-value,Outperformance Count,Outperformance Rate (%),Binomial P-value
0,6M Issue Price,190,18.957738,7.374011,6577.0,0.001010,107,56.315789,0.094938
1,6M Listing Close,185,0.861744,-5.577930,7507.0,0.133080,78,42.162162,0.039244
2,12M Issue Price,190,27.934833,5.641294,6755.0,0.002263,101,53.157895,0.424932
3,12M Listing Close,185,7.706885,-9.401440,7951.0,0.371704,76,41.081081,0.018396
